# 5-1 손실 함수 심화

강의 원문 대신 직접 작성한 코드, 실행 결과와 학습 메모를 정리했습니다.


In [4]:
# 검증 가능 정답 코드
import torch
pred = torch.tensor([[1.], [2.], [3.]])
target = torch.tensor([1., 2., 4.])
# (3,1)-(3,)은 오류 없이 (3,3)으로 broadcast되므로 scalar loss보다 먼저 diff shape를 감사합니다.
buggy_diff = pred - target
buggy_loss = (buggy_diff ** 2).mean()
# 샘플당 target 하나라는 계약에 맞춰 마지막 축만 추가하고 loss 전에 shape 동일성을 고정합니다.
target = target.view(-1,1)
assert pred.shape == target.shape
fixed_diff = pred - target
fixed_loss = (fixed_diff ** 2).mean()
print("shapes:", (buggy_diff.shape), (fixed_diff.shape))
print("losses:", f"{buggy_loss.item():.4f}", f"{fixed_loss.item():.4f}")

shapes: torch.Size([3, 3]) torch.Size([3, 1])
losses: 2.3333 0.3333


In [10]:
# 검증 가능 정답 코드
import torch
from torch import nn
pred = torch.tensor([[1.], [2.], [3.]])
target = torch.tensor([[1.], [4.], [2.]])
# MSE는 차이를 먼저 제곱한 뒤 reduction하므로 원소별 제곱 오차를 중간 결과로 보존합니다.
squared = (pred - target) ** 2
# 같은 squared Tensor에서 mean과 sum을 만들고 PyTorch의 두 reduction과 각각 대조합니다.
# squared tensor를 기준으로 mean, sum 만들고 torch.allclose()하기
manual_mean, manual_sum = squared.mean(), squared.sum()
mean_loss = nn.MSELoss(reduction="mean")(pred, target)
sum_loss = nn.MSELoss(reduction="sum")(pred, target)
print("squared:", squared.squeeze(1).tolist())
print("mean_sum:", f"{manual_mean.item():.4f}", f"{manual_sum.item():.4f}")
print("matches:", torch.allclose(manual_mean, mean_loss) and torch.allclose(manual_sum, sum_loss))

squared: [0.0, 4.0, 1.0]
mean_sum: 1.6667 5.0000
matches: True


In [11]:
# 검증 가능 정답 코드
import torch
target = torch.tensor([1., 2., 4.])
candidates = {"A": torch.tensor([1., 2., 3.]), "B": torch.tensor([1., 2.5, 4.])}
# 두 후보는 같은 target·단위·MSE mean 계약에서만 비교해 loss 숫자의 의미를 통제합니다.
losses = {name: float(((pred - target) ** 2).mean()) for name, pred in candidates.items()}
# 공통 계약에서 계산된 loss 중 최소값을 고르되 이 선택을 곧바로 배포 승인으로 확대하지 않습니다.
selected = min(losses, key=losses.get) # key = losses.get <<다시보기
print("losses:", {n: round(v, 4) for n, v in losses.items()})
print("selected:", selected)

losses: {'A': 0.3333, 'B': 0.0833}
selected: B
